In [ ]:
%pip install --quiet pymilvus feast transformers sentence-transformers

In [1]:
import urllib.request

link = "https://huggingface.co/ngxson/demo_simple_rag_py/raw/main/cat-facts.txt"
dataset = []

# Retrieve knowledge from provided link, use every line as a separate chunk.
for line in urllib.request.urlopen(link):
  dataset.append(line.decode('utf-8'))

print(f'Loaded {len(dataset)} entries')

Loaded 150 entries


In [2]:
from sentence_transformers import SentenceTransformer

# load pretrained model sentence transformer model (for testing - to be replaced later)
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(dataset)

print(f"Generated embeddings of shape: {embeddings.shape}")


Generated embeddings of shape: (150, 384)


In [3]:
import pandas as pd
from datetime import datetime

# Create DataFrame
df = pd.DataFrame({
    "cat_fact_id": list(range(len(dataset))),
    "fact": dataset,
    "embedding": pd.Series(list(embeddings), dtype=object),
    "event_timestamp": [datetime.utcnow() for _ in dataset],
})

# Save to Parquet
df.to_parquet("cat_facts.parquet", index=False)
print("Saved to cat_facts.parquet")

Saved to cat_facts.parquet


In [8]:
from pymilvus import connections

# connect to milvus 
connections.connect(alias="hello", host="vectordb-milvus.milvus.svc.cluster.local", port="19530", timeout=10, user="root", password="Milvus")

# Check connection status 
print("Connected:", connections.has_connection(alias="hello"))

Connected: True


In [9]:
# Create collection in Milvus
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility

# Drop the old collection if it exists
if utility.has_collection("cat_facts"):
    Collection("cat_facts").drop()

# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]
schema = CollectionSchema(fields)
collection = Collection(name="cat_facts", schema=schema, using="hello")

# Create index
collection.create_index(
    field_name="embedding",
    index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 64}}
)

collection.load()


ConnectionNotExistException: <ConnectionNotExistException: (code=1, message=should create connection first.)>

In [ ]:
# Insert embeddings into Milvus
import numpy as np

ids = list(range(len(dataset)))
vectors = embeddings.tolist()

collection.insert([ids, vectors])
collection.flush()

print(f"Inserted {len(ids)} cat fact embeddings into Milvus.")


In [ ]:
%pwd

In [ ]:
# Initialise Feature Store (Feast)
!feast init fionaProject

In [ ]:
%pwd

In [ ]:
%cd fionaProject/feature_repo
!feast apply

In [ ]:
from transformers import AutoTokenizer, AutoConfig

model = SentenceTransformer("all-MiniLM-L6-v2")

question_encoder_tokenizer = model.tokenizer

generator_tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.2-2b-instruct")
generator_config = AutoConfig.from_pretrained("ibm-granite/granite-3.2-2b-instruct")

In [ ]:
from abc import ABC, abstractmethod
from pymilvus import Collection
from transformers import RagRetriever
import numpy as np


class VectorStore(ABC):
    @abstractmethod
    def query(self, query_vector: np.ndarray, top_k: int):
        pass


class MilvusVectorStore(VectorStore):
    def __init__(self, collection_name: str):
        self.collection = Collection(name=collection_name, using="fiona")

    def query(self, query_vector: np.ndarray, top_k: int = 5):
        results = self.collection.search(
            data=[query_vector],
            anns_field="vector",
            param={"metric_type": "L2", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["doc_id"]
        )
        return results[0]  # return the top-k results list


class FeastRAGRetriever(RagRetriever):
    def __init__(
            self, 
            question_encoder_tokenizer, 
            generator_tokenizer, 
            feast_repo_path: str, 
            vector_store, 
            config,
            **kwargs
    ):
        super().__init__(question_encoder_tokenizer=question_encoder_tokenizer,
                          generator_tokenizer=generator_tokenizer, config=config,
                          **kwargs)
        self.feast = FeatureStore(repo_path=feast_repo_path)
        self.vector_store = vector_store

    def retrieve(self, query_vector: np.ndarray, top_k: int) -> list[str]:
        store_results = self.vector_store.query(query_vector, top_k=top_k)

        doc_ids = [results["doc_id"] for results in store_results if "doc_id" in results]
        entity_rows = [{"doc_id": doc_id} for doc_id in doc_ids]

        feast_docs = self.feast.retrieve_online_documents(
            entity_rows=entity_rows,
            features=[
                # feature details
            ],
            top_k=top_k,
        )

        formatted_docs = [
            self.format_document({
                "doc_id": doc_ids[i],
                "feast_features": feast_docs[i],
                **store_results[i]  
            })
            for i in range(len(doc_ids))
        ]

        return formatted_docs

    def format_document(self, doc: dict) -> str:
        # format the returned document/s into format required for RAG

        features = doc.get("feast_features", {})
        title = features.get("title", "")
        body = features.get("body", "")

        return f"Title: {title}\nBody: {body}"


In [ ]:
%pip install datasets --quiet

In [ ]:
# Initialize Milvus Vector Store
milvus_store = MilvusVectorStore(collection_name="cat_facts")

# Initialize the FeastRAGRetriever
rag_retriever = FeastRAGRetriever(
    question_encoder_tokenizer=question_encoder_tokenizer,
    generator_tokenizer=generator_tokenizer,
    feast_repo_path=".",
    vector_store=milvus_store,
    config=generator_config,
)

# Retrieve top-k documents
top_k_documents = rag_retriever.retrieve(embeddings, top_k=5)

# Print the top-k retrieved documents
for doc in top_k_documents:
    print(doc)
